# Interactive Exploration of the OctreeHybridMesherModeler

This notebook demonstrates every stage of the `KratosMultiphysics.OctreeHybridMesherModeler`
pipeline using simple Python-constructed surface meshes and PyVista for 3-D visualisation.

**Pipeline overview** — `SetupModelPart` executes four sequential stages:

| Stage | JSON key | What it does |
|-------|----------|--------------|
| 1 | `octree_generator` | Builds & 2:1-balances the adaptive octree, extracts the dual or primal hex mesh. |
| 2 | `coloring_settings_list` | Classifies cells as inside (1) or outside (0) the input surface. |
| 3 | `entities_generator_list` | Emits hexahedral elements, surface conditions, and/or hanging-node constraints. |
| 4 | `model_part_operations` | Post-processing passes (e.g. mesh-quality statistics). |

**Components covered:**
- `ClassifyCellsInsideOutside` — inside/outside colouring via ray-cast + signed distance.
- `GenerateHexesByCellColor` — one `Element3D8N` per cell matching a colour value.
- `GenerateBoundaryConditionsByFace` — `SurfaceCondition3D4N` on the outer boundary.
- `GenerateHangingNodeConstraints` — `LinearMasterSlaveConstraint` for primal-mesh 2:1 transitions.
- `ReportMeshQuality` — scaled-Jacobian statistics logged to the Kratos output stream.

### Enabling Interactive 3-D Rendering

By default this notebook renders **static** PNG images.  To enable interactive rotate/pan/zoom:
1. Install the required packages:
   ```bash
   pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
   ```
2. Un-comment and run the interactive backend lines in the imports cell below.

In [ ]:
import sys
import os

# Resolve the Kratos repo root relative to this notebook's location:
# notebooks/ -> python_scripts/ -> kratos/ -> <repo root>
# NOTE: Only needed for self-compiled Kratos; skip for system-wide installations.
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
kratos_root = os.path.abspath(os.path.join(notebook_dir, "..", "..", ".."))
kratos_build_path = os.path.join(kratos_root, "bin", "Release")

if kratos_build_path not in sys.path:
    sys.path.insert(0, kratos_build_path)

print(f"Kratos build path: {kratos_build_path}")

In [ ]:
import collections
import struct
import tempfile

import numpy as np
import pyvista as pv
import KratosMultiphysics as KM
import KratosMultiphysics.pyvista_utilities as pv_utils

pv.global_theme.color = "white"

# ==========================================================================
# INTERACTIVE 3-D VISUALISATION IN JUPYTER NOTEBOOKS:
# 1. pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
# 2. Un-comment the lines below:
#    import nest_asyncio
#    nest_asyncio.apply()
#    pv.set_jupyter_backend("client")
# ==========================================================================

print("Modules imported successfully!")

### Step 1: Build Input Surface Geometry

The modeler requires a **closed triangulated surface** stored as `Triangle3D3` geometries inside
a Kratos `ModelPart`.  Two helper functions are defined here:

- **`build_closed_box_surface`** — a simple axis-aligned cube `[lo, hi]³`, triangulated into
  12 triangles (2 per face).  Two extra bounding-box pin nodes (at the unit-cube corners) are
  added as required by the octree engine.
- **`build_transition_surface`** — a small inclined quad patch near one corner of the unit cube.
  Being far smaller than the domain, it forces multiple levels of 2:1 refinement in the adaptive
  octree — useful for testing the primal mesh and hanging-node constraints.

> **Note:** The two bounding-box pin nodes (`[0,0,0]` and `[1,1,1]`) tell the octree engine the
> extent of the mesh domain.  They do **not** need to be part of any triangle.

In [ ]:
def build_closed_box_surface(model, lo=0.3, hi=0.7, name="Surface"):
    """Closed triangulated cube [lo, hi]^3 with two bounding-box pin nodes."""
    mp = model.CreateModelPart(name)
    mp.ProcessInfo[KM.DOMAIN_SIZE] = 3

    corners = [
        (lo, lo, lo), (hi, lo, lo), (hi, hi, lo), (lo, hi, lo),
        (lo, lo, hi), (hi, lo, hi), (hi, hi, hi), (lo, hi, hi),
    ]
    for i, (x, y, z) in enumerate(corners, start=1):
        mp.CreateNewNode(i, x, y, z)
    mp.CreateNewNode(9,  0.0, 0.0, 0.0)   # bbox pin (min)
    mp.CreateNewNode(10, 1.0, 1.0, 1.0)   # bbox pin (max)

    # 12 triangles — 2 per cube face (0-indexed corner offsets)
    faces = [
        (0,1,2),(0,2,3),  # -Z face
        (4,6,5),(4,7,6),  # +Z face
        (0,5,1),(0,4,5),  # -Y face
        (3,2,6),(3,6,7),  # +Y face
        (0,3,7),(0,7,4),  # -X face
        (1,5,6),(1,6,2),  # +X face
    ]
    for gid, (a, b, c) in enumerate(faces, start=1):
        mp.CreateNewGeometry("Triangle3D3", gid, [a+1, b+1, c+1])
    return mp


def build_transition_surface(model, name="Surface"):
    """Small inclined patch that forces 2:1 transitions in the adaptive octree."""
    mp = model.CreateModelPart(name)
    mp.ProcessInfo[KM.DOMAIN_SIZE] = 3

    pts = [
        (0.0, 0.0, 0.0), (1.0, 1.0, 1.0),   # bbox pins
        (0.15, 0.15, 0.30), (0.45, 0.15, 0.30),
        (0.45, 0.45, 0.36), (0.15, 0.45, 0.36),
    ]
    for i, (x, y, z) in enumerate(pts, start=1):
        mp.CreateNewNode(i, x, y, z)
    mp.CreateNewGeometry("Triangle3D3", 1, [3, 4, 5])
    mp.CreateNewGeometry("Triangle3D3", 2, [3, 5, 6])
    return mp


# Build the primary working surface for Steps 2–5
model = KM.Model()
build_closed_box_surface(model, lo=0.3, hi=0.7, name="Surface")
print(f"Surface model part: {model.GetModelPart('Surface').NumberOfNodes()} nodes, "
      f"{model.GetModelPart('Surface').NumberOfGeometries()} triangles")

### Step 2: Dual Hex Mesh — Minimal Pipeline

The *dual* mesh is the default topology: the octree is 2:1-balanced, and the dual of each
interior primal vertex becomes one conforming hexahedron.  Transition templates stitch cells
at refinement boundaries so the mesh has **no hanging nodes**.

The minimal pipeline needs:
1. **`ClassifyCellsInsideOutside`** — marks interior cells with colour `1`.
2. **`GenerateHexesByCellColor`** — emits `Element3D8N` for every `color=1` cell.

Setting `adaptive: false` and a fixed `refinement_depth` yields a **uniform** octree —
every cell is the same size, all hexes are perfect cuboids, and the mesh is easy to inspect.

In [ ]:
settings_dual = KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "output_model_part_name" : "Volume",
    "octree_generator" : {
        "type"            : "generate_octree_from_surface",
        "refinement_depth": 4,
        "adaptive"        : false,
        "mesh_type"       : "dual"
    },
    "coloring_settings_list"  : [{ "type": "ClassifyCellsInsideOutside" }],
    "entities_generator_list" : [{
        "type"               : "GenerateHexesByCellColor",
        "model_part_name"    : "Volume",
        "color"              : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : []
}
""")

mod = KM.OctreeHybridMesherModeler(model, settings_dual)
mod.SetupModelPart()

mp_vol = model.GetModelPart("Volume")
print(f"Dual hex mesh (uniform, depth=4):")
print(f"  Nodes    : {mp_vol.NumberOfNodes()}")
print(f"  Elements : {mp_vol.NumberOfElements()}")

# Quick quality check: no inverted elements
SJ_ADJ = [(1,3,4),(2,0,5),(3,1,6),(0,2,7),(7,5,0),(4,6,1),(5,7,2),(6,4,3)]

def _sub(p, q): return (p[0]-q[0], p[1]-q[1], p[2]-q[2])
def _norm(v):   return (v[0]**2 + v[1]**2 + v[2]**2)**0.5
def _dot(a, b): return a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
def _cross(a, b): return (a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0])

def min_scaled_jacobian(element):
    geom = element.GetGeometry()
    coords = [(geom[i].X, geom[i].Y, geom[i].Z) for i in range(8)]
    worst = 1e30
    for o, x, y, z in SJ_ADJ:
        e1 = _sub(coords[x], coords[o])
        e2 = _sub(coords[y], coords[o])
        e3 = _sub(coords[z], coords[o])
        n1, n2, n3 = _norm(e1), _norm(e2), _norm(e3)
        if min(n1, n2, n3) < 1e-14:
            return -1.0
        worst = min(worst, _dot(e1, _cross(e2, e3)) / (n1 * n2 * n3))
    return worst

n_inverted = sum(1 for el in mp_vol.Elements if min_scaled_jacobian(el) <= 0)
all_sj = [min_scaled_jacobian(el) for el in mp_vol.Elements]
print(f"  Inverted elements  : {n_inverted}")
print(f"  Min scaled Jacobian: {min(all_sj):.6f}")
print(f"  Mean scaled Jacobian: {sum(all_sj)/len(all_sj):.6f}")

In [ ]:
# Visualise the dual hex mesh.
# Since the generated nodes carry no simulation variables, we add a synthetic
# point field (distance from the domain centre) for colouring.
grid = pv_utils.ModelPartToPyVista(mp_vol)

cx, cy, cz = 0.5, 0.5, 0.5
grid.point_data["dist_to_centre"] = np.array(
    [((n.X - cx)**2 + (n.Y - cy)**2 + (n.Z - cz)**2)**0.5 for n in mp_vol.Nodes]
)

grid.plot(
    scalars="dist_to_centre",
    show_edges=True,
    cmap="viridis",
    cpos="iso",
    text="Dual Hex Mesh — carved box [0.3, 0.7]³",
)

### Step 3: Inside/Outside Colouring — What `ClassifyCellsInsideOutside` Does

`ClassifyCellsInsideOutside` classifies every octree cell by a ray-cast parity test combined
with a closest-triangle signed distance.  Cells with colour `1` are *inside* the surface;
colour `0` cells are *outside*.

The cell below runs the pipeline **with** and **without** the colouring stage to show how many
cells are removed when the inside-only filter is applied.

In [ ]:
# --- Unfiltered: no colouring, all cells emitted (color=1 default means all pass) ---
model_all = KM.Model()
build_closed_box_surface(model_all, name="Surface")
mod_all = KM.OctreeHybridMesherModeler(model_all, KM.Parameters("""
{
    "input_model_part_name" : "Surface",
    "output_model_part_name": "All",
    "octree_generator": {"refinement_depth": 4, "adaptive": false},
    "coloring_settings_list" : [],
    "entities_generator_list": [{"type": "GenerateHexesByCellColor",
                                  "model_part_name": "All", "color": 1}],
    "model_part_operations"  : []
}
"""))
mod_all.SetupModelPart()
n_all = model_all.GetModelPart("All").NumberOfElements()

# --- Carved: ClassifyCellsInsideOutside removes outside cells ---
model_carved = KM.Model()
build_closed_box_surface(model_carved, name="Surface")
mod_carved = KM.OctreeHybridMesherModeler(model_carved, KM.Parameters("""
{
    "input_model_part_name" : "Surface",
    "output_model_part_name": "Carved",
    "octree_generator": {"refinement_depth": 4, "adaptive": false},
    "coloring_settings_list" : [{"type": "ClassifyCellsInsideOutside"}],
    "entities_generator_list": [{"type": "GenerateHexesByCellColor",
                                  "model_part_name": "Carved", "color": 1}],
    "model_part_operations"  : []
}
"""))
mod_carved.SetupModelPart()
n_inside = model_carved.GetModelPart("Carved").NumberOfElements()

print(f"Total octree cells (no coloring) : {n_all}")
print(f"Inside cells (color=1)           : {n_inside}")
print(f"Outside cells removed            : {n_all - n_inside}")
print(f"Carve ratio (inside/total)       : {n_inside / n_all:.1%}")

# Side-by-side visualisation
grid_all    = pv_utils.ModelPartToPyVista(model_all.GetModelPart("All"))
grid_carved = pv_utils.ModelPartToPyVista(model_carved.GetModelPart("Carved"))

pl = pv.Plotter(shape=(1, 2))
pl.subplot(0, 0)
pl.add_mesh(grid_all,    color="steelblue",  show_edges=True)
pl.add_text("All cells (no coloring)", font_size=10)
pl.subplot(0, 1)
pl.add_mesh(grid_carved, color="tomato",     show_edges=True)
pl.add_text("Inside cells only",            font_size=10)
pl.link_views()
pl.show()

### Step 4: Boundary Conditions

`GenerateBoundaryConditionsByFace` scans every hex cell coloured `1` and emits one
`SurfaceCondition3D4N` per quad face that is **owned by exactly one hex** — i.e. faces on the
outer boundary of the carved region.

The conditions are placed in a separate `ModelPart` (`"Boundary"`) so they can be assigned
boundary values independently from the volume mesh.  Every boundary node is also present in the
volume `ModelPart` (shared node IDs), so no duplication occurs.

In [ ]:
model_bc = KM.Model()
build_closed_box_surface(model_bc, name="Surface")

mod_bc = KM.OctreeHybridMesherModeler(model_bc, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "output_model_part_name" : "Volume",
    "octree_generator" : {"refinement_depth": 4, "adaptive": false},
    "coloring_settings_list"  : [{"type": "ClassifyCellsInsideOutside"}],
    "entities_generator_list" : [
        {"type": "GenerateHexesByCellColor",
         "model_part_name": "Volume",   "color": 1},
        {"type": "GenerateBoundaryConditionsByFace",
         "model_part_name": "Boundary", "color": 1}
    ],
    "model_part_operations" : []
}
"""))
mod_bc.SetupModelPart()

mp_vol_bc = model_bc.GetModelPart("Volume")
mp_bnd    = model_bc.GetModelPart("Boundary")

print(f"Volume  : {mp_vol_bc.NumberOfElements()} hexes,  {mp_vol_bc.NumberOfNodes()} nodes")
print(f"Boundary: {mp_bnd.NumberOfConditions()} quads,  {mp_bnd.NumberOfNodes()} nodes")
print(f"Max possible boundary faces (6 × n_elem)  : {6 * mp_vol_bc.NumberOfElements()}")
print(f"Actual boundary faces                     : {mp_bnd.NumberOfConditions()}")
print(f"  (ratio < 1 confirms interior faces are shared)")

# Check: every boundary node id is also in the volume mesh
vol_ids = {n.Id for n in mp_vol_bc.Nodes}
bnd_ids = {n.Id for n in mp_bnd.Nodes}
assert bnd_ids.issubset(vol_ids), "Boundary nodes not a subset of volume nodes!"
print("Node-id consistency check: PASSED (boundary ⊆ volume)")

In [ ]:
# Visualise: volume wireframe + extracted outer surface coloured tomato
vol_grid_bc = pv_utils.ModelPartToPyVista(mp_vol_bc)
outer_surf  = pv_utils.CreateExtractedSurface(mp_vol_bc)

plotter_bc = pv.Plotter()
plotter_bc.add_mesh(vol_grid_bc, style="wireframe", color="steelblue",
                    opacity=0.35, label="Volume hexes")
plotter_bc.add_mesh(outer_surf,  color="tomato", show_edges=True,
                    opacity=0.85, label="Outer surface")
plotter_bc.add_legend()
plotter_bc.add_text("Volume mesh + outer boundary surface", font_size=11)
plotter_bc.show()

### Step 5: Adaptive Mesh and Quality Report

Setting `"adaptive": true` allows the octree engine to selectively refine cells near the input
surface, producing a graded mesh: fine near the surface, coarser in the interior.  This reduces
the element count compared to a uniform octree of the same maximum depth.

With `"tag_refinement_level": true`, every element stores its octree level in the non-historical
variable `REFINEMENT_LEVEL` (accessed via `element.GetValue(KM.REFINEMENT_LEVEL)`).  Template
hexes at refinement transitions receive the sentinel value `−1`.

The `ReportMeshQuality` operation logs minimum scaled-Jacobian statistics at the end of the pipeline.

In [ ]:
model_adapt = KM.Model()
build_closed_box_surface(model_adapt, name="Surface")

mod_adapt = KM.OctreeHybridMesherModeler(model_adapt, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "output_model_part_name" : "Volume",
    "octree_generator" : {
        "type"            : "generate_octree_from_surface",
        "refinement_depth": 5,
        "adaptive"        : true,
        "mesh_type"       : "dual"
    },
    "coloring_settings_list"  : [{"type": "ClassifyCellsInsideOutside"}],
    "entities_generator_list" : [{
        "type"                : "GenerateHexesByCellColor",
        "model_part_name"     : "Volume",
        "color"               : 1,
        "tag_refinement_level": true
    }],
    "model_part_operations" : [
        {"type": "ReportMeshQuality", "model_part_name": "Volume"}
    ]
}
"""))
mod_adapt.SetupModelPart()

mp_adapt = model_adapt.GetModelPart("Volume")
print(f"Adaptive dual mesh (depth=5):")
print(f"  Nodes    : {mp_adapt.NumberOfNodes()}")
print(f"  Elements : {mp_adapt.NumberOfElements()}")

levels = [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_adapt.Elements]
level_dist = dict(sorted(collections.Counter(levels).items()))
print(f"  REFINEMENT_LEVEL distribution: {level_dist}")
print(f"    (−1 = transition-template hex, positive = leaf level)")

In [ ]:
# Visualise adaptive mesh coloured by REFINEMENT_LEVEL (cell data)
grid_adapt = pv_utils.ModelPartToPyVista(mp_adapt)
grid_adapt.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_adapt.Elements]
)

grid_adapt.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text="Adaptive Dual Mesh — coloured by octree refinement level",
)

### Step 6: Primal Mesh with Hanging-Node Constraints

Setting `"mesh_type": "primal"` produces one hexahedron per octree **leaf cell** (rather than per
primal vertex).  This is faster to generate and avoids the dual extraction + transition-template
pass, but the mesh is **non-conforming** at 2:1 refinement boundaries: finer cells introduce
*hanging nodes* that lie on the face or edge of a coarser neighbour without being one of its
corner nodes.

`GenerateHangingNodeConstraints` resolves these by emitting a `LinearMasterSlaveConstraint` for
each hanging DOF:

$$u_s = \sum_{m} w_m \, u_m$$

where the bilinear interpolation weights $w_m$ satisfy the **partition-of-unity** property
$\sum_m w_m = 1$.  Hanging nodes on an edge have 2 masters; face-centre hanging nodes have 4.

In [ ]:
# Use the transition surface to guarantee 2:1 refinement transitions
model_primal = KM.Model()
build_transition_surface(model_primal, name="Surface")

mod_primal = KM.OctreeHybridMesherModeler(model_primal, KM.Parameters("""
{
    "input_model_part_name"  : "Surface",
    "output_model_part_name" : "Volume",
    "octree_generator" : {
        "type"            : "generate_octree_from_surface",
        "refinement_depth": 4,
        "adaptive"        : true,
        "mesh_type"       : "primal"
    },
    "coloring_settings_list"  : [],
    "entities_generator_list" : [
        {
            "type"               : "GenerateHexesByCellColor",
            "model_part_name"    : "Volume",
            "color"              : 1,
            "tag_refinement_level": true
        },
        {
            "type"            : "GenerateHangingNodeConstraints",
            "model_part_name" : "Volume",
            "variables"       : ["DISPLACEMENT_X", "DISPLACEMENT_Y", "DISPLACEMENT_Z"]
        }
    ],
    "model_part_operations" : []
}
"""))
mod_primal.SetupModelPart()

mp_primal = model_primal.GetModelPart("Volume")
nc = mp_primal.NumberOfMasterSlaveConstraints()
print(f"Primal mesh (adaptive, depth=4):")
print(f"  Nodes       : {mp_primal.NumberOfNodes()}")
print(f"  Elements    : {mp_primal.NumberOfElements()}")
print(f"  Constraints : {nc}  "
      f"({nc // 3} hanging nodes × 3 DOF variables)")

In [ ]:
# Verify partition-of-unity and master-count distribution
n_fail       = 0
master_counts = []

for c in mp_primal.MasterSlaveConstraints:
    T, b = KM.Matrix(), KM.Vector()
    c.CalculateLocalSystem(T, b, KM.ProcessInfo())
    row_sum = sum(T[0, j] for j in range(T.Size2()))
    master_counts.append(T.Size2())
    if abs(row_sum - 1.0) > 1e-10:
        n_fail += 1

mc_dist = dict(sorted(collections.Counter(master_counts).items()))
print(f"Partition-of-unity violations: {n_fail} / {nc}")
print(f"Master-count distribution    : {mc_dist}")
print(f"  2-master = edge-midpoint hanging node")
print(f"  4-master = face-centre  hanging node")

In [ ]:
# Visualise primal mesh coloured by REFINEMENT_LEVEL
grid_primal = pv_utils.ModelPartToPyVista(mp_primal)
grid_primal.cell_data["REFINEMENT_LEVEL"] = np.array(
    [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_primal.Elements]
)

grid_primal.plot(
    scalars="REFINEMENT_LEVEL",
    show_edges=True,
    cmap="tab10",
    cpos="iso",
    text="Primal Mesh — refinement level (non-conforming at transitions)",
)

### Step 7: Dual Mesh — Conformity Comparison

A useful sanity check is to run **both** the dual and primal topologies on the same surface
and compare element counts.  For the same `refinement_depth`:

- The **primal** mesh has exactly one hex per leaf cell (leaf count = element count).
- The **dual** mesh has one hex per interior primal vertex — fewer cells in a uniform octree,
  but additional transition-template hexes near boundaries mean the counts can differ.

The dual mesh never creates hanging-node constraints; the primal mesh always does when
`adaptive=true`.

In [ ]:
def run_and_count(mesh_type, adaptive, depth=4):
    """Build a transition-surface mesh and return element/constraint counts."""
    m = KM.Model()
    build_transition_surface(m, name="S")
    generators = [
        '{"type":"GenerateHexesByCellColor","model_part_name":"O","color":1}',
    ]
    coloring = '[{"type":"ClassifyCellsInsideOutside"}]' if mesh_type == "dual" else '[]'
    if mesh_type == "primal":
        generators.append(
            '{"type":"GenerateHangingNodeConstraints","model_part_name":"O",'
            '"variables":["DISPLACEMENT_X"]}'
        )
    gen_json = "[" + ",".join(generators) + "]"
    adaptive_str = "true" if adaptive else "false"
    settings = KM.Parameters(f"""{{
        "input_model_part_name":"S","output_model_part_name":"O",
        "octree_generator":{{"refinement_depth":{depth},
                              "adaptive":{adaptive_str},"mesh_type":"{mesh_type}"}},
        "coloring_settings_list":{coloring},
        "entities_generator_list":{gen_json},
        "model_part_operations":[]
    }}""")
    KM.OctreeHybridMesherModeler(m, settings).SetupModelPart()
    out = m.GetModelPart("O")
    return out.NumberOfElements(), out.NumberOfMasterSlaveConstraints()


for depth in (3, 4, 5):
    n_dual,   nc_dual   = run_and_count("dual",   adaptive=True,  depth=depth)
    n_primal, nc_primal = run_and_count("primal", adaptive=True,  depth=depth)
    print(f"depth={depth}  |  dual: {n_dual:5d} elems, {nc_dual:4d} constraints  "
          f"|  primal: {n_primal:5d} elems, {nc_primal:4d} constraints")

### Step 8: Loading a Surface from STL (Optional)

Any closed, orientable surface can be loaded from an STL file using `KM.StlIO`.  The cell below
looks for `Bunny-LowPoly.stl` in the Kratos test directory; if found it runs the full dual-mesh
pipeline and reports statistics.  The test is skipped gracefully when the file is absent.

Both ASCII and binary STL files are accepted — the helper converts binary files on the fly.

In [ ]:
def load_stl_surface(model, stl_path, name="Surface"):
    """Load an ASCII or binary STL surface into a new ModelPart."""
    # Detect binary STL (does not start with the ASCII 'solid' keyword)
    with open(stl_path, "rb") as f:
        header = f.read(80)
    is_binary = not header.lstrip().decode("ascii", errors="replace").startswith("solid")

    ascii_path = stl_path
    tmp_path   = None
    if is_binary:
        tmp_path = tempfile.mktemp(suffix=".stl")
        with open(stl_path, "rb") as f:
            f.read(80)
            n = struct.unpack("<I", f.read(4))[0]
            with open(tmp_path, "w") as out:
                out.write("solid s\n")
                for _ in range(n):
                    f.read(12)  # normal
                    out.write("facet normal 0 0 1\n outer loop\n")
                    for _ in range(3):
                        x, y, z = struct.unpack("<fff", f.read(12))
                        out.write(f"  vertex {x} {y} {z}\n")
                    f.read(2)
                    out.write(" endloop\nendfacet\n")
                out.write("endsolid s\n")
        ascii_path = tmp_path

    mp = model.CreateModelPart(name)
    mp.ProcessInfo[KM.DOMAIN_SIZE] = 3
    KM.StlIO(ascii_path, KM.Parameters('{"open_mode":"read"}')).ReadModelPart(mp)
    if tmp_path and os.path.exists(tmp_path):
        os.remove(tmp_path)
    return mp


bunny_path = os.path.join(
    notebook_dir, "..", "..", "tests",
    "auxiliar_files_for_python_unittest", "stl_files", "Bunny-LowPoly.stl"
)

if os.path.exists(bunny_path):
    model_stl = KM.Model()
    mp_stl = load_stl_surface(model_stl, bunny_path, name="Bunny")
    print(f"Loaded STL: {mp_stl.NumberOfNodes()} nodes, "
          f"{mp_stl.NumberOfGeometries()} triangles")

    mod_stl = KM.OctreeHybridMesherModeler(model_stl, KM.Parameters("""
    {
        "input_model_part_name"  : "Bunny",
        "output_model_part_name" : "BunnyVolume",
        "octree_generator" : {
            "refinement_depth": 4,
            "adaptive"        : true,
            "mesh_type"       : "dual"
        },
        "coloring_settings_list"  : [{"type": "ClassifyCellsInsideOutside"}],
        "entities_generator_list" : [{
            "type"                : "GenerateHexesByCellColor",
            "model_part_name"     : "BunnyVolume",
            "color"               : 1,
            "tag_refinement_level": true
        }],
        "model_part_operations" : [
            {"type": "ReportMeshQuality", "model_part_name": "BunnyVolume"}
        ]
    }
    """))
    mod_stl.SetupModelPart()

    mp_bunny = model_stl.GetModelPart("BunnyVolume")
    print(f"Bunny hex mesh: {mp_bunny.NumberOfElements()} elements, "
          f"{mp_bunny.NumberOfNodes()} nodes")

    grid_bunny = pv_utils.ModelPartToPyVista(mp_bunny)
    grid_bunny.cell_data["REFINEMENT_LEVEL"] = np.array(
        [el.GetValue(KM.REFINEMENT_LEVEL) for el in mp_bunny.Elements]
    )
    grid_bunny.plot(
        scalars="REFINEMENT_LEVEL",
        cmap="tab10",
        show_edges=True,
        cpos="iso",
        text="Bunny Hex Mesh — adaptive dual, depth=4",
    )
else:
    print(f"Bunny-LowPoly.stl not found at:\n  {os.path.normpath(bunny_path)}")
    print("Provide any closed STL surface and update 'bunny_path' to run this cell.")

### Step 9: Exporting to VTU / VTK

Two export paths are available:

1. **PyVista `.save()`** — writes the `pyvista.UnstructuredGrid` directly to a `.vtu` file,
   including any point- or cell-data arrays that were added (e.g. `REFINEMENT_LEVEL`).
2. **`pv_utils.SaveModelPart`** — Kratos-native helper that converts the `ModelPart` to a grid
   and writes it, preserving any nodal `Variable` arrays listed in `nodalVariables`.

Both formats can be opened directly in **ParaView**.

In [ ]:
temp_dir = tempfile.gettempdir()

# --- Option 1: save the PyVista grid (includes REFINEMENT_LEVEL cell data) ---
vtu_pv = os.path.join(temp_dir, "hex_mesh_adaptive.vtu")
grid_adapt.save(vtu_pv)
print(f"PyVista grid saved to : {vtu_pv}")

# --- Option 2: Kratos SaveModelPart (mesh geometry only, no nodal variables here) ---
vtu_km = os.path.join(temp_dir, "hex_mesh_modelpart.vtu")
pv_utils.SaveModelPart(mp_adapt, vtu_km)
print(f"Kratos ModelPart saved: {vtu_km}")

# --- Option 3: low-level OctreeHybridMeshUtility VTK writer (legacy format) ---
vtk_util = os.path.join(temp_dir, "hex_mesh_utility.vtk")
model_vtk = KM.Model()
build_closed_box_surface(model_vtk, name="Surface")
KM.OctreeHybridMeshUtility.BuildAndWriteVtk(
    model_vtk.GetModelPart("Surface"), vtk_util, 4
)
print(f"Utility VTK saved to  : {vtk_util}")
print("\nAll three files can be opened in ParaView.")
print("Colour by 'REFINEMENT_LEVEL' (option 1) or 'level' (option 3) to see adaptive refinement.")